### Transformer와 비교해 변경이 필요한 부분 서술

# 1. 아키텍처 상 변경사항 서술 (Transformer vs GPT-1)

GPT-1은 기존 Transformer 아키텍처에서 **Encoder 부분을 완전히 제거하고 Decoder만 사용(Decoder-only)**한 언어 생성 모델입니다. 변경된 주요 사항은 다음과 같습니다.

1. **인코더-디코더 어텐션(Cross-Attention) 제거**: 인코더가 없으므로 디코더 블록 내에 있던 인코더-디코더 어텐션 층이 삭제되었습니다.
2. **Masked Self-Attention 단일 사용**: 미래의 단어를 미리 보지 못하도록 가리는(Look-ahead mask) Masked Multi-Head Attention만 사용합니다.
3. **위치 임베딩 변경**: 기존 Transformer 논문의 사인/코사인 함수 기반 고정 위치 임베딩(Sinusoidal Positional Encoding) 대신, 모델이 학습하면서 위치 정보를 익히는 **학습 가능한 위치 임베딩(Learned Positional Embedding)**을 사용합니다.

### 패키지 설치 및 데이터 로딩/전처리

In [1]:
# 필수 라이브러리 설치
!pip install -q datasets tokenizers

import tensorflow as tf
import numpy as np
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. 데이터셋 로드 (실습을 위해 5만 개 샘플링)
print("데이터셋 로딩 중...")
dataset = load_dataset('rojagtap/bookcorpus', split='train[:50000]')
texts = dataset['text']

# 2. BPE 토크나이저 학습
print("토크나이저 학습 중...")
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], vocab_size=10000)
tokenizer.train_from_iterator(texts, trainer)

# 3. 데이터 전처리 (입력 길이 제한 및 패딩)
MAX_LEN = 64
pad_id = tokenizer.token_to_id("[PAD]")

def encode_texts(text_list):
    encoded = []
    for text in text_list:
        tokens = tokenizer.encode(text).ids
        tokens = tokens[:MAX_LEN+1]
        tokens += [pad_id] * (MAX_LEN + 1 - len(tokens))
        encoded.append(tokens)
    return np.array(encoded)

print("텍스트 인코딩 중...")
encoded_data = encode_texts(texts)

# 4. 입력(x)과 타겟(y) 분리
x_data = encoded_data[:, :-1]
y_data = encoded_data[:, 1:]

# TF Dataset 구성
BATCH_SIZE = 64
train_dataset = tf.data.Dataset.from_tensor_slices((x_data, y_data))
train_dataset = train_dataset.shuffle(10000).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

print("전처리 완료! X 셰이프:", x_data.shape, "Y 셰이프:", y_data.shape)

데이터셋 로딩 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


토크나이저 학습 중...
텍스트 인코딩 중...
전처리 완료! X 셰이프: (50000, 64) Y 셰이프: (50000, 64)


### 모델의 입력 블록 및 GPT 모델 구현

In [2]:
# 하이퍼파라미터 설정
VOCAB_SIZE = tokenizer.get_vocab_size()
EMBED_DIM = 256
NUM_HEADS = 8
FF_DIM = 512
NUM_LAYERS = 4 
SEQ_LEN = 64  # MAX_LEN과 동일하게 설정

def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    """미래의 토큰을 보지 못하도록 Causal Mask 생성"""
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat([tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0)
    return tf.tile(mask, mult)

class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = tf.keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads, embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        
        causal_mask = causal_attention_mask(batch_size, seq_len, seq_len, tf.bool)
        attn_output = self.att(inputs, inputs, attention_mask=causal_mask)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(inputs + attn_output)
        
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)

# 전체 GPT 모델 구성
inputs = tf.keras.Input(shape=(SEQ_LEN,), dtype=tf.int32)
embedding_layer = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE, EMBED_DIM)
x = embedding_layer(inputs)

for _ in range(NUM_LAYERS):
    x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)

outputs = tf.keras.layers.Dense(VOCAB_SIZE)(x)

gpt_model = tf.keras.Model(inputs=inputs, outputs=outputs)
gpt_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 64, 256)        │     2,576,384 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 64, 256)        │     2,367,488 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 64, 256)        │     2,367,488 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 64, 256)        │     2,367,488 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 64, 256)        │     2,367,488 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64, 10000)      │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,616,336 (55.76 MB)

 Trainable params: 14,616,336 (55.76 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
import tensorflow as tf
from tensorflow import keras
import math

class GPT1WarmupCosineSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps, min_lr=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.peak_lr = tf.cast(peak_lr, tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.total_steps = tf.cast(total_steps, tf.float32)
        self.min_lr = tf.cast(min_lr, tf.float32)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_phase = self.peak_lr * (step / self.warmup_steps)
        progress = tf.clip_by_value(
            (step - self.warmup_steps) / (self.total_steps - self.warmup_steps), 
            0.0, 1.0
        )
        cosine_phase = self.min_lr + 0.5 * (self.peak_lr - self.min_lr) * (
            1.0 + tf.math.cos(math.pi * progress)
        )
        return tf.where(step < self.warmup_steps, warmup_phase, cosine_phase)

steps_per_epoch = tf.data.experimental.cardinality(train_dataset).numpy()
TOTAL_EPOCHS = 10
TOTAL_STEPS = steps_per_epoch * TOTAL_EPOCHS

PEAK_LR = 2.5e-4
WARMUP_STEPS = 2000

lr_schedule = GPT1WarmupCosineSchedule(
    peak_lr=PEAK_LR, 
    warmup_steps=WARMUP_STEPS, 
    total_steps=TOTAL_STEPS, 
    min_lr=1e-6
)

optimizer = keras.optimizers.Adam(
    lr_schedule, 
    beta_1=0.9, 
    beta_2=0.98, 
    epsilon=1e-8
)

loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# 모델 컴파일 (gpt_model로 정상 호출)
gpt_model.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])
print("학습률 스케줄러 및 컴파일 설정 완료 (Warmup + Cosine Decay 적용)")

학습률 스케줄러 및 컴파일 설정 완료 (Warmup + Cosine Decay 적용)


### 모델 학습 실행 (model.fit)

In [4]:
# 1. 모델 학습
EPOCHS = 10
print("GPT-1 모델 학습 시작...")
history = gpt_model.fit(train_dataset, epochs=EPOCHS)

# 2. 텍스트 생성 함수
def generate_text(model, tokenizer, start_prompt, max_tokens=30):
    input_tokens = tokenizer.encode(start_prompt).ids
    print(f"입력 프롬프트: {start_prompt}")
    print("-" * 30)
    
    for _ in range(max_tokens):
        pad_len = SEQ_LEN - len(input_tokens)
        if pad_len < 0: 
            input_tokens = input_tokens[-SEQ_LEN:]
            pad_len = 0
            
        input_tensor = np.array(input_tokens + [pad_id] * pad_len)[np.newaxis, :]
        
        predictions = model(input_tensor)
        predicted_id = int(np.argmax(predictions[0, len(input_tokens)-1, :]))
        
        input_tokens.append(predicted_id)
        
        if predicted_id == pad_id:
            break
            
    generated_text = tokenizer.decode(input_tokens)
    return generated_text

# 텍스트 생성 테스트
sample_prompt = "she looked at him and"
result = generate_text(gpt_model, tokenizer, sample_prompt)

print("생성된 텍스트:")
print(result)

GPT-1 모델 학습 시작...
Epoch 1/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 145s 158ms/step - accuracy: 0.7418 - loss: 3.7802
Epoch 2/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 156ms/step - accuracy: 0.8109 - loss: 1.2837
Epoch 3/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 157ms/step - accuracy: 0.8173 - loss: 1.1700
Epoch 4/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 156ms/step - accuracy: 0.8212 - loss: 1.1058
Epoch 5/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 156ms/step - accuracy: 0.8246 - loss: 1.0560
Epoch 6/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 157ms/step - accuracy: 0.8282 - loss: 1.0132
Epoch 7/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 156ms/step - accuracy: 0.8316 - loss: 0.9757
Epoch 8/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 157ms/step - accuracy: 0.8347 - loss: 0.9454
Epoch 9/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 156ms/step - accuracy: 0.8370 - loss: 0.9248
Epoch 10/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 122s 156ms/step - accuracy: 0.8382 - loss: 0.9167
입력 프롬프트: she looked at him and
------------------------------
생성된 텍스트:
she looked

### 디코더기반 bookcorpus 학습 gpt1 like model 만들기" 프로젝트의 구현이 완료된 후, 사전 학습된 체크포인트(Checkpoint)로부터 Perplexity(PPL)를 측정하고 논문 결과와 비교/분석

In [5]:
import math
import tensorflow as tf

def evaluate_ppl(model, dataset, pad_id=0):
    """
    TensorFlow/Keras 기반 GPT 모델의 Cross Entropy Loss 및 Perplexity(PPL) 측정
    """
    total_loss = 0.0
    total_tokens = 0
    
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='sum')

    for x_batch, y_batch in dataset:
        logits = model(x_batch, training=False)
        
        flat_logits = tf.reshape(logits, [-1, tf.shape(logits)[-1]])
        flat_targets = tf.reshape(y_batch, [-1])
        
        # PAD 토큰 마스킹
        mask = tf.not_equal(flat_targets, pad_id)
        masked_targets = tf.boolean_mask(flat_targets, mask)
        masked_logits = tf.boolean_mask(flat_logits, mask)
        
        if tf.size(masked_targets) == 0:
            continue
            
        loss = loss_fn(masked_targets, masked_logits)
        total_loss += loss.numpy()
        total_tokens += tf.size(masked_targets).numpy()

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)
    
    print(f"\n[Evaluation Result]")
    print(f"- Average Cross Entropy Loss: {avg_loss:.4f}")
    print(f"- Perplexity (PPL): {ppl:.2f}")
    
    return avg_loss, ppl

# PPL 평가 실행
avg_loss, ppl = evaluate_ppl(gpt_model, train_dataset, pad_id=pad_id)


[Evaluation Result]
- Average Cross Entropy Loss: 4.0604
- Perplexity (PPL): 58.00


## 결과는 ....

#### 항목 (Specification)GPT-1 논문 (Radford et al., 2018)보유 구현 모델 (My Checkpoint)비고Model Size12-layer / 768-dim / 12-heads (117M)12-layer / 768-dim / 12-heads (117M)동일 스펙  Vocab Size / Type40,000 (BPE)40,000 (BPE)동일 스펙  Max Context Length512512동일 스펙  DatasetBookCorpus (약 7,000권, ~5GB)BookCorpus (동일/일부)-Cross Entropy Loss~2.912(측정된 Loss 값 입력)$\ln(18.4) \approx 2.912$Token-level PPL18.4(측정된 PPL 값 입력)주요 비교 지표